In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


## Подготовка данных к анализу в STATA

In [5]:
panel = pd.read_excel('data/final_data.xlsx')
panel["rep_date"] = pd.to_datetime(panel["rep_date"])
panel["country"] = panel["country"].astype(str).str.strip()
panel["hs"] = panel["hs"].astype(str).str.strip().str.zfill(4)

num_cols = [
    "value",
    "sanctions_proxy",
    "sanctions_proxy_smooth",
    "cpi_yoy",
    "ip_yoy",
    "ex_yoy",
    "gscpi",
    "distw",
    "logistics_exposure_distw",
    "unfriendly",
    "brics",
    "cis",
    "post_sanctions",
    "unfriendly_post",
    "brics_post",
    "cis_post",
]

for col in num_cols:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce")

panel = panel.sort_values(["country", "hs", "rep_date"]).reset_index(drop=True)
panel

,rep_date,country,hs,value,sanctions_proxy,sanctions_proxy_smooth,cpi_yoy,ip_yoy,ex_yoy,gscpi,distw,logistics_exposure_distw,unfriendly,brics,cis,post_sanctions,unfriendly_post,brics_post,cis_post
0,2019-01-01,Albania,9018,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,2777.355,4.086917,1,0,0,0,0,0,0
1,2019-02-01,Albania,9018,0.0,0,0,1.716270,5.800000,2.086924,0.145187,2777.355,1.151223,1,0,0,0,0,0,0
2,2019-03-01,Albania,9018,0.0,0,0,1.107924,5.800000,3.863434,0.216385,2777.355,1.715773,1,0,0,0,0,0,0
3,2019-04-01,Albania,9018,0.0,0,0,1.351889,5.900000,5.080519,0.017049,2777.355,0.135183,1,0,0,0,0,0,0
4,2019-05-01,Albania,9018,0.0,0,0,1.529021,6.000000,2.264640,-0.623563,2777.355,-4.944387,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48433,2025-02-01,Uzbekistan,9031,194528.0,0,0,9.888332,7.456653,3.862195,-0.002936,2528.859,-0.023007,0,0,1,1,0,0,1
48434,2025-03-01,Uzbekistan,9031,472578.0,0,0,10.104674,7.492429,2.989072,-0.185382,2528.859,-1.452565,0,0,1,1,0,0,1
48435,2025-04-01,Uzbekistan,9031,60124.0,0,0,9.891634,5.938055,2.176561,-0.249671,2528.859,-1.956305,0,0,1,1,0,0,1
48436,2025-05-01,Uzbekistan,9031,0.0,0,0,8.497777,6.885872,1.696777,0.276993,2528.859,2.170384,0,0,1,1,0,0,1


#### Проверка пропусков

In [6]:
missing_table = panel.isna().sum().rename("n_missing").to_frame()
missing_table["share_missing"] = missing_table["n_missing"] / len(panel)
print(missing_table.sort_values("n_missing", ascending=False))

                          n_missing  share_missing
rep_date                          0            0.0
distw                             0            0.0
brics_post                        0            0.0
unfriendly_post                   0            0.0
post_sanctions                    0            0.0
cis                               0            0.0
brics                             0            0.0
unfriendly                        0            0.0
logistics_exposure_distw          0            0.0
gscpi                             0            0.0
country                           0            0.0
ex_yoy                            0            0.0
ip_yoy                            0            0.0
cpi_yoy                           0            0.0
sanctions_proxy_smooth            0            0.0
sanctions_proxy                   0            0.0
value                             0            0.0
hs                                0            0.0
cis_post                       

#### ID ДЛЯ STATA

In [7]:
country_codes = (
    panel[["country"]]
    .drop_duplicates()
    .sort_values("country")
    .reset_index(drop=True)
)
country_codes["country_id"] = np.arange(1, len(country_codes) + 1)
panel = panel.merge(country_codes, on="country", how="left")

hs_codes = (
    panel[["hs"]]
    .drop_duplicates()
    .sort_values("hs")
    .reset_index(drop=True)
)
hs_codes["hs_id"] = np.arange(1, len(hs_codes) + 1)
panel = panel.merge(hs_codes, on="hs", how="left")

month_codes = (
    panel[["rep_date"]]
    .drop_duplicates()
    .sort_values("rep_date")
    .reset_index(drop=True)
)
month_codes["month_id"] = np.arange(1, len(month_codes) + 1)
panel = panel.merge(month_codes, on="rep_date", how="left")

country_hs_codes = (
    panel[["country_id", "hs_id"]]
    .drop_duplicates()
    .sort_values(["country_id", "hs_id"])
    .reset_index(drop=True)
)
country_hs_codes["country_hs_id"] = np.arange(1, len(country_hs_codes) + 1)
panel = panel.merge(country_hs_codes, on=["country_id", "hs_id"], how="left")
panel

,rep_date,country,hs,value,sanctions_proxy,sanctions_proxy_smooth,cpi_yoy,ip_yoy,ex_yoy,gscpi,...,brics,cis,post_sanctions,unfriendly_post,brics_post,cis_post,country_id,hs_id,month_id,country_hs_id
0,2019-01-01,Albania,9018,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,...,0,0,0,0,0,0,1,1,1,1
1,2019-02-01,Albania,9018,0.0,0,0,1.716270,5.800000,2.086924,0.145187,...,0,0,0,0,0,0,1,1,2,1
2,2019-03-01,Albania,9018,0.0,0,0,1.107924,5.800000,3.863434,0.216385,...,0,0,0,0,0,0,1,1,3,1
3,2019-04-01,Albania,9018,0.0,0,0,1.351889,5.900000,5.080519,0.017049,...,0,0,0,0,0,0,1,1,4,1
4,2019-05-01,Albania,9018,0.0,0,0,1.529021,6.000000,2.264640,-0.623563,...,0,0,0,0,0,0,1,1,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48433,2025-02-01,Uzbekistan,9031,194528.0,0,0,9.888332,7.456653,3.862195,-0.002936,...,0,1,1,0,0,1,69,9,74,621
48434,2025-03-01,Uzbekistan,9031,472578.0,0,0,10.104674,7.492429,2.989072,-0.185382,...,0,1,1,0,0,1,69,9,75,621
48435,2025-04-01,Uzbekistan,9031,60124.0,0,0,9.891634,5.938055,2.176561,-0.249671,...,0,1,1,0,0,1,69,9,76,621
48436,2025-05-01,Uzbekistan,9031,0.0,0,0,8.497777,6.885872,1.696777,0.276993,...,0,1,1,0,0,1,69,9,77,621


#### STATA DATE

In [8]:
panel["year"] = panel["rep_date"].dt.year
panel["month"] = panel["rep_date"].dt.month
panel["stata_mdate"] = (panel["year"] - 1960) * 12 + (panel["month"] - 1)

#### Лаги

In [9]:
country_month_base = (
    panel[
        [
            "country",
            "country_id",
            "rep_date",
            "month_id",
            "year",
            "month",
            "stata_mdate",
            "sanctions_proxy",
            "sanctions_proxy_smooth",
            "cpi_yoy",
            "ip_yoy",
            "ex_yoy",
            "gscpi",
            "distw",
            "logistics_exposure_distw",
            "unfriendly",
            "brics",
            "cis",
            "post_sanctions",
            "unfriendly_post",
            "brics_post",
            "cis_post",
        ]
    ]
    .drop_duplicates()
    .sort_values(["country_id", "rep_date"])
    .reset_index(drop=True)
)

lag_vars = [
    "sanctions_proxy",
    "sanctions_proxy_smooth",
    "cpi_yoy",
    "ip_yoy",
    "ex_yoy",
    "gscpi",
    "logistics_exposure_distw",
]

for var in lag_vars:
    for L in [1, 2, 3, 4, 5]:
        country_month_base[f"{var}_l{L}"] = (
            country_month_base.groupby("country_id")[var].shift(L)
        )

lag_cols = [c for c in country_month_base.columns if c.endswith(("_l1", "_l2", "_l3", "_l4", "_l5"))]

panel = panel.merge(
    country_month_base[["country_id", "rep_date"] + lag_cols],
    on=["country_id", "rep_date"],
    how="left"
)

panel

,rep_date,country,hs,value,sanctions_proxy,sanctions_proxy_smooth,cpi_yoy,ip_yoy,ex_yoy,gscpi,...,gscpi_l1,gscpi_l2,gscpi_l3,gscpi_l4,gscpi_l5,logistics_exposure_distw_l1,logistics_exposure_distw_l2,logistics_exposure_distw_l3,logistics_exposure_distw_l4,logistics_exposure_distw_l5
0,2019-01-01,Albania,9018,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-02-01,Albania,9018,0.0,0,0,1.716270,5.800000,2.086924,0.145187,...,0.515423,NaN,NaN,NaN,NaN,4.086917,NaN,NaN,NaN,NaN
2,2019-03-01,Albania,9018,0.0,0,0,1.107924,5.800000,3.863434,0.216385,...,0.145187,0.515423,NaN,NaN,NaN,1.151223,4.086917,NaN,NaN,NaN
3,2019-04-01,Albania,9018,0.0,0,0,1.351889,5.900000,5.080519,0.017049,...,0.216385,0.145187,0.515423,NaN,NaN,1.715773,1.151223,4.086917,NaN,NaN
4,2019-05-01,Albania,9018,0.0,0,0,1.529021,6.000000,2.264640,-0.623563,...,0.017049,0.216385,0.145187,0.515423,NaN,0.135183,1.715773,1.151223,4.086917,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48433,2025-02-01,Uzbekistan,9031,194528.0,0,0,9.888332,7.456653,3.862195,-0.002936,...,-0.181087,-0.264668,-0.283382,-0.349705,0.101886,-1.418908,-2.073812,-2.220448,-2.740125,0.798331
48434,2025-03-01,Uzbekistan,9031,472578.0,0,0,10.104674,7.492429,2.989072,-0.185382,...,-0.002936,-0.181087,-0.264668,-0.283382,-0.349705,-0.023007,-1.418908,-2.073812,-2.220448,-2.740125
48435,2025-04-01,Uzbekistan,9031,60124.0,0,0,9.891634,5.938055,2.176561,-0.249671,...,-0.185382,-0.002936,-0.181087,-0.264668,-0.283382,-1.452565,-0.023007,-1.418908,-2.073812,-2.220448
48436,2025-05-01,Uzbekistan,9031,0.0,0,0,8.497777,6.885872,1.696777,0.276993,...,-0.249671,-0.185382,-0.002936,-0.181087,-0.264668,-1.956305,-1.452565,-0.023007,-1.418908,-2.073812


#### Панель country-month

In [10]:
country_month = (
    panel.groupby(
        [
            "country",
            "country_id",
            "rep_date",
            "month_id",
            "year",
            "month",
            "stata_mdate",
        ],
        as_index=False
    )
    .agg(
        value=("value", "sum"),
        sanctions_proxy=("sanctions_proxy", "first"),
        sanctions_proxy_smooth=("sanctions_proxy_smooth", "first"),
        cpi_yoy=("cpi_yoy", "first"),
        ip_yoy=("ip_yoy", "first"),
        ex_yoy=("ex_yoy", "first"),
        gscpi=("gscpi", "first"),
        distw=("distw", "first"),
        logistics_exposure_distw=("logistics_exposure_distw", "first"),
        unfriendly=("unfriendly", "first"),
        brics=("brics", "first"),
        cis=("cis", "first"),
        post_sanctions=("post_sanctions", "first"),
        unfriendly_post=("unfriendly_post", "first"),
        brics_post=("brics_post", "first"),
        cis_post=("cis_post", "first"),
    )
    .sort_values(["country_id", "rep_date"])
    .reset_index(drop=True)
)

for var in lag_vars:
    for L in [1, 2, 3, 4, 5]:
        country_month[f"{var}_l{L}"] = (
            country_month.groupby("country_id")[var].shift(L)
        )

In [11]:
country_month

,country,country_id,rep_date,month_id,year,month,stata_mdate,value,sanctions_proxy,sanctions_proxy_smooth,...,gscpi_l1,gscpi_l2,gscpi_l3,gscpi_l4,gscpi_l5,logistics_exposure_distw_l1,logistics_exposure_distw_l2,logistics_exposure_distw_l3,logistics_exposure_distw_l4,logistics_exposure_distw_l5
0,Albania,1,2019-01-01,1,2019,1,708,0.0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,1,2019-02-01,2,2019,2,709,0.0,0,0,...,0.515423,NaN,NaN,NaN,NaN,4.086917,NaN,NaN,NaN,NaN
2,Albania,1,2019-03-01,3,2019,3,710,0.0,0,0,...,0.145187,0.515423,NaN,NaN,NaN,1.151223,4.086917,NaN,NaN,NaN
3,Albania,1,2019-04-01,4,2019,4,711,0.0,0,0,...,0.216385,0.145187,0.515423,NaN,NaN,1.715773,1.151223,4.086917,NaN,NaN
4,Albania,1,2019-05-01,5,2019,5,712,0.0,0,0,...,0.017049,0.216385,0.145187,0.515423,NaN,0.135183,1.715773,1.151223,4.086917,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5377,Uzbekistan,69,2025-02-01,74,2025,2,781,314744.0,0,0,...,-0.181087,-0.264668,-0.283382,-0.349705,0.101886,-1.418908,-2.073812,-2.220448,-2.740125,0.798331
5378,Uzbekistan,69,2025-03-01,75,2025,3,782,717943.0,0,0,...,-0.002936,-0.181087,-0.264668,-0.283382,-0.349705,-0.023007,-1.418908,-2.073812,-2.220448,-2.740125
5379,Uzbekistan,69,2025-04-01,76,2025,4,783,333198.0,0,0,...,-0.185382,-0.002936,-0.181087,-0.264668,-0.283382,-1.452565,-0.023007,-1.418908,-2.073812,-2.220448
5380,Uzbekistan,69,2025-05-01,77,2025,5,784,0.0,0,0,...,-0.249671,-0.185382,-0.002936,-0.181087,-0.264668,-1.956305,-1.452565,-0.023007,-1.418908,-2.073812


#### ПАНЕЛЬ country-hs-month

In [12]:
country_hs_month = (
    panel.sort_values(["country_id", "hs_id", "rep_date"])
    .drop_duplicates(subset=["country_id", "hs_id", "rep_date"])
    .reset_index(drop=True)
)
country_hs_month

,rep_date,country,hs,value,sanctions_proxy,sanctions_proxy_smooth,cpi_yoy,ip_yoy,ex_yoy,gscpi,...,gscpi_l1,gscpi_l2,gscpi_l3,gscpi_l4,gscpi_l5,logistics_exposure_distw_l1,logistics_exposure_distw_l2,logistics_exposure_distw_l3,logistics_exposure_distw_l4,logistics_exposure_distw_l5
0,2019-01-01,Albania,9018,0.0,0,0,1.930000,5.800000,-0.163976,0.515423,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-02-01,Albania,9018,0.0,0,0,1.716270,5.800000,2.086924,0.145187,...,0.515423,NaN,NaN,NaN,NaN,4.086917,NaN,NaN,NaN,NaN
2,2019-03-01,Albania,9018,0.0,0,0,1.107924,5.800000,3.863434,0.216385,...,0.145187,0.515423,NaN,NaN,NaN,1.151223,4.086917,NaN,NaN,NaN
3,2019-04-01,Albania,9018,0.0,0,0,1.351889,5.900000,5.080519,0.017049,...,0.216385,0.145187,0.515423,NaN,NaN,1.715773,1.151223,4.086917,NaN,NaN
4,2019-05-01,Albania,9018,0.0,0,0,1.529021,6.000000,2.264640,-0.623563,...,0.017049,0.216385,0.145187,0.515423,NaN,0.135183,1.715773,1.151223,4.086917,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48433,2025-02-01,Uzbekistan,9031,194528.0,0,0,9.888332,7.456653,3.862195,-0.002936,...,-0.181087,-0.264668,-0.283382,-0.349705,0.101886,-1.418908,-2.073812,-2.220448,-2.740125,0.798331
48434,2025-03-01,Uzbekistan,9031,472578.0,0,0,10.104674,7.492429,2.989072,-0.185382,...,-0.002936,-0.181087,-0.264668,-0.283382,-0.349705,-0.023007,-1.418908,-2.073812,-2.220448,-2.740125
48435,2025-04-01,Uzbekistan,9031,60124.0,0,0,9.891634,5.938055,2.176561,-0.249671,...,-0.185382,-0.002936,-0.181087,-0.264668,-0.283382,-1.452565,-0.023007,-1.418908,-2.073812,-2.220448
48436,2025-05-01,Uzbekistan,9031,0.0,0,0,8.497777,6.885872,1.696777,0.276993,...,-0.249671,-0.185382,-0.002936,-0.181087,-0.264668,-1.956305,-1.452565,-0.023007,-1.418908,-2.073812


#### Служебные переменные

In [13]:
country_month["value_log1p"] = np.log1p(country_month["value"])
country_hs_month["value_log1p"] = np.log1p(country_hs_month["value"])

country_month["is_zero_value"] = (country_month["value"] == 0).astype(int)
country_hs_month["is_zero_value"] = (country_hs_month["value"] == 0).astype(int)

#### Сохранение в STATA

In [14]:
print("country_month shape:", country_month.shape)
print("country_hs_month shape:", country_hs_month.shape)

print("\nДоля нулей в country_month:", (country_month["value"] == 0).mean())
print("Доля нулей в country_hs_month:", (country_hs_month["value"] == 0).mean())

print("\nЧисло стран:", country_month["country_id"].nunique())
print("Число HS:", country_hs_month["hs_id"].nunique())
print("Число месяцев:", country_month["rep_date"].nunique())

country_month shape: (5382, 60)
country_hs_month shape: (48438, 63)

Доля нулей в country_month: 0.21330360460795245
Доля нулей в country_hs_month: 0.5529542920847269

Число стран: 69
Число HS: 9
Число месяцев: 78


In [15]:
country_month.to_stata(
    "data/country_month_panel.dta",
    write_index=False,
    version=118
)

country_hs_month.to_stata(
    "data/country_hs_month_panel.dta",
    write_index=False,
    version=118
)